# Customer Intelligence System: Classification, Ensemble & Clustering**Objective:** Build an end-to-end Customer Intelligence System that segments entities into actionable groups using unsupervised learning (K-Means, DBSCAN), and generalizes those segments with supervised ensemble classifiers (Random Forest, XGBoost).**Dataset note:** This project uses the Kaggle *Country socio-economic* dataset (child mortality, income, GDP per capita, health spend, exports/imports, etc.) as a stand-in for "customer" records. Each row (a country) is treated as a customer/account profile, and the same clustering + classification pipeline used here for macro-segmentation applies directly to customer segmentation on business data with analogous numeric features (spend, engagement, tenure, etc.).

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python# For example, here's several helpful packages to loadimport numpy as np # linear algebraimport pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)# Input data files are available in the read-only "../input/" directory# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directoryimport osfor dirname, _, filenames in os.walk('/kaggle/input'):    for filename in filenames:        print(os.path.join(dirname, filename))# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.mdimport kagglehub# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.preprocessing import StandardScalerfrom sklearn.cluster import KMeans, DBSCANfrom sklearn.decomposition import PCAfrom sklearn.model_selection import train_test_splitfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrixfrom xgboost import XGBClassifierimport warningswarnings.filterwarnings('ignore')sns.set_style('whitegrid')

## 1. Load Data

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/rohan0301/unsupervised-learning-on-country-data/Country-data.csv')df.head()

## 2. Exploratory Data Analysis

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

**Data quality check:** No missing values across any column, and all features are numeric except `country`. This means no imputation is required before scaling and modeling — we can move straight into correlation and distribution analysis.

In [ ]:
plt.figure(figsize=(10,6))sns.heatmap(df.drop('country', axis=1).corr(), annot=True, cmap='coolwarm', fmt='.2f')plt.title('Correlation Heatmap of Features')plt.show()

**Correlation insight:** `child_mort` is strongly negatively correlated with `income`, `gdpp`, and `life_expec`, while `income` and `gdpp` are strongly positively correlated with each other. This tells us these features carry overlapping signal about economic development — exactly the kind of structure clustering algorithms can exploit to separate customers/countries into meaningful tiers.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))sns.histplot(df['gdpp'], kde=True, ax=axes[0])axes[0].set_title('GDP per capita distribution')sns.histplot(df['child_mort'], kde=True, ax=axes[1])axes[1].set_title('Child Mortality distribution')sns.histplot(df['income'], kde=True, ax=axes[2])axes[2].set_title('Income distribution')plt.tight_layout()plt.show()

**Distribution insight:** `gdpp` and `income` are both right-skewed, with a long tail of high-income outliers — a small number of very developed countries pull the distribution. `child_mort` is also right-skewed, meaning most countries cluster at low mortality with a smaller group at high mortality. This skew is exactly why we scale features before clustering (K-Means and DBSCAN are distance-based and sensitive to differing feature ranges).

## 3. Clustering — Customer/Country Segmentation

### 3.1 K-Means

In [ ]:
features = df.drop('country', axis=1)scaler = StandardScaler()scaled_features = scaler.fit_transform(features)scaled_df = pd.DataFrame(scaled_features, columns=features.columns)scaled_df.head()

In [ ]:
wcss = []for k in range(1, 10):    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)    kmeans.fit(scaled_features)    wcss.append(kmeans.inertia_)plt.figure(figsize=(8,5))plt.plot(range(1, 10), wcss, marker='o')plt.xlabel('Number of Clusters (k)')plt.ylabel('WCSS')plt.title('Elbow Method to find optimal k')plt.show()

**Elbow method result:** WCSS drops sharply up to k=3 and flattens after that, so k=3 is a reasonable choice — it balances segment granularity against diminishing returns in cluster tightness.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)df['cluster_kmeans'] = kmeans.fit_predict(scaled_features)df['cluster_kmeans'].value_counts()

In [ ]:
df.groupby('cluster_kmeans')[['child_mort', 'income', 'gdpp', 'life_expec']].mean()

In [ ]:
cluster_order = df.groupby('cluster_kmeans')['gdpp'].mean().sort_values().index.tolist()label_map = {cluster_order[0]: 'Under-developed', cluster_order[1]: 'Developing', cluster_order[2]: 'Developed'}df['development_label'] = df['cluster_kmeans'].map(label_map)df[['country', 'gdpp', 'income', 'child_mort', 'cluster_kmeans', 'development_label']].head(10)

In [ ]:
df['development_label'].value_counts()

**K-Means segment insight:** The three clusters separate cleanly along `gdpp`: the **Under-developed** segment shows high child mortality and low income/GDP, the **Developing** segment sits in the middle on all three metrics, and the **Developed** segment has low child mortality and high income/GDP. In a customer-intelligence context, these three tiers map directly onto something like Low-Value / Growth / High-Value customer segments — each would warrant a different engagement or retention strategy.

In [ ]:
pca = PCA(n_components=2)pca_features = pca.fit_transform(scaled_features)plt.figure(figsize=(8,6))sns.scatterplot(x=pca_features[:,0], y=pca_features[:,1], hue=df['development_label'], palette='Set2')plt.xlabel('PCA Component 1')plt.ylabel('PCA Component 2')plt.title('Country Clusters (K-Means) visualized using PCA')plt.show()

### 3.2 DBSCAN — Density-Based Comparison

In [ ]:
dbscan = DBSCAN(eps=1.2, min_samples=5)df['cluster_dbscan'] = dbscan.fit_predict(scaled_features)df['cluster_dbscan'].value_counts()

In [ ]:
n_noise = (df['cluster_dbscan'] == -1).sum()n_clusters_dbscan = df['cluster_dbscan'].nunique() - (1 if -1 in df['cluster_dbscan'].values else 0)print(f'DBSCAN found {n_clusters_dbscan} dense cluster(s) and flagged {n_noise} countries as noise/outliers.')df.groupby('cluster_dbscan')[['child_mort', 'income', 'gdpp', 'life_expec']].mean()

In [ ]:
plt.figure(figsize=(8,6))sns.scatterplot(x=pca_features[:,0], y=pca_features[:,1], hue=df['cluster_dbscan'], palette='Set1')plt.xlabel('PCA Component 1')plt.ylabel('PCA Component 2')plt.title('Country Clusters (DBSCAN) visualized using PCA')plt.show()

**K-Means vs DBSCAN insight:** K-Means forces every point into one of 3 evenly-shaped segments, which is useful for building clean, actionable tiers. DBSCAN instead finds one dense core group and flags the rest as noise/outliers — those noise points are typically the extreme, unusual countries (very high or very low on income/mortality) that don't fit a typical density profile. In a real customer-intelligence pipeline, DBSCAN's "noise" points are valuable in their own right: they're the atypical customers (e.g. very high spenders or high-risk churners) that a rigid K-Means segment would otherwise blend into an average bucket. We use the K-Means labels for classification below because they give evenly-populated, more actionable segments, while DBSCAN serves as an outlier-detection layer on top.

## 4. Classification — Generalizing Segments with Ensemble Models

**Note on methodology:** `development_label` was derived *from* K-Means clusters on these same features, so the classifiers below are learning to reproduce that clustering rule rather than predicting an independent ground truth. This is a standard **cluster-then-classify** pattern — it's done deliberately so that once trained, the classifier can assign a segment to *new* countries/customers instantly, without having to re-run clustering on the full dataset each time. Expect high accuracy here since the label is feature-derived; the real value is the reusable classifier, not the accuracy score.

In [ ]:
X = features  # original (unscaled is fine for tree-based models)y = df['development_label']X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)print('Train size:', X_train.shape)print('Test size:', X_test.shape)

### 4.1 Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)rf_model.fit(X_train, y_train)rf_pred = rf_model.predict(X_test)print('Random Forest Accuracy:', accuracy_score(y_test, rf_pred))print()print(classification_report(y_test, rf_pred))

In [ ]:
cm = confusion_matrix(y_test, rf_pred, labels=rf_model.classes_)plt.figure(figsize=(6,5))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=rf_model.classes_, yticklabels=rf_model.classes_)plt.xlabel('Predicted')plt.ylabel('Actual')plt.title('Confusion Matrix - Random Forest')plt.show()

**Random Forest result:** As expected from the cluster-then-classify setup, Random Forest reproduces the K-Means-derived segments with very high accuracy, confirming the segment boundaries are cleanly separable in feature space — a good sign that the underlying clusters are meaningful and not arbitrary.

### 4.2 XGBoost

In [ ]:
from sklearn.preprocessing import LabelEncoderle = LabelEncoder()y_train_enc = le.fit_transform(y_train)y_test_enc = le.transform(y_test)xgb_model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')xgb_model.fit(X_train, y_train_enc)xgb_pred = xgb_model.predict(X_test)print('XGBoost Accuracy:', accuracy_score(y_test_enc, xgb_pred))print()print(classification_report(y_test_enc, xgb_pred, target_names=le.classes_))

In [ ]:
cm_xgb = confusion_matrix(y_test_enc, xgb_pred)plt.figure(figsize=(6,5))sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Greens', xticklabels=le.classes_, yticklabels=le.classes_)plt.xlabel('Predicted')plt.ylabel('Actual')plt.title('Confusion Matrix - XGBoost')plt.show()

**Random Forest vs XGBoost:** Both ensemble models perform comparably well on this task, which is expected given the clean, low-noise separation between the K-Means segments. In a production customer-intelligence system with noisier real-world behavioral data, XGBoost's boosting approach would typically pull ahead of Random Forest's bagging approach on harder-to-separate segments.

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)plt.figure(figsize=(8,5))sns.barplot(x=importances.values, y=importances.index)plt.title('Feature Importance - Random Forest')plt.xlabel('Importance')plt.show()

**Feature importance insight:** `gdpp` and `income` dominate the classifier's decisions, consistent with them being the features used to order and label the K-Means clusters. `child_mort` is the next strongest signal — an inverse indicator of development. For a real customer dataset, this step is what tells the business *which* metrics actually drive segment membership (e.g. spend vs. tenure vs. support tickets), which is directly actionable for prioritizing what to track per customer.

## 5. Actionable Insights Summary- **Three clear segments emerged** (Under-developed / Developing / Developed), separated primarily by `gdpp`, `income`, and `child_mort` — in a business setting these map to Low / Mid / High-value customer tiers, each warranting a distinct engagement strategy (e.g. retention offers for the low tier, upsell campaigns for the mid tier, loyalty programs for the high tier).- **DBSCAN surfaced outliers** that K-Means blends into an average segment — these atypical records deserve manual review or a dedicated "VIP/at-risk" handling path rather than being folded into a standard tier.- **Both ensemble classifiers (Random Forest, XGBoost) generalize the segmentation rule** with high accuracy, meaning new records can be auto-assigned a segment in real time without re-running clustering on the full dataset — this is the piece that makes the system production-usable rather than a one-off analysis.- **Feature importance confirms `gdpp`, `income`, and `child_mort`-equivalent metrics should be the first fields captured** for any new record entering the system, since they carry the most segmentation signal.